# LECTURE 21 — DEMONSTRATION
### MODULE 9: INTERPRETABILITY, GOVERNANCE & COMMUNICATION

**DEMONSTRATION** — Model Interpretability with SHAP Values

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, r2_score,
    roc_auc_score, classification_report,
)

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print(f"XGBoost : {xgb.__version__}")
print(f"SHAP    : {shap.__version__}")

## ── SECTION 2: SYNTHETIC MACRO DATA

In [ ]:
def generate_macro_forecast_data(n_obs: int = 500, seed: int = 42) -> pd.DataFrame:
    """
    500 country-year observations for predicting next-year inflation.

    Features:
      current_inflation, money_growth, gdp_growth, output_gap,
      exchange_rate_chg, oil_price_chg, interest_rate,
      fiscal_deficit, external_debt, credit_growth
    Target: next_year_inflation (continuous regression target)
    """
    rng = np.random.default_rng(seed)

    current_inflation  = rng.gamma(2.5, 3.5, n_obs)
    money_growth       = rng.normal(12, 4, n_obs)
    gdp_growth         = rng.normal(3.5, 2, n_obs)
    output_gap         = rng.normal(0, 1.5, n_obs)
    exchange_rate_chg  = rng.normal(0, 5, n_obs)
    oil_price_chg      = rng.normal(0, 12, n_obs)
    interest_rate      = rng.normal(8, 3, n_obs).clip(1, 25)
    fiscal_deficit     = rng.normal(-3, 2, n_obs)
    external_debt      = rng.normal(45, 15, n_obs).clip(5, 110)
    credit_growth      = rng.normal(12, 6, n_obs)

    next_inflation = (
        0.60 * current_inflation
        + 0.30 * money_growth
        + 0.12 * oil_price_chg
        + 0.15 * exchange_rate_chg
        - 0.18 * gdp_growth
        - 0.10 * interest_rate
        + 0.05 * fiscal_deficit
        + 0.04 * external_debt * 0.1
        + 0.08 * credit_growth
        + rng.normal(0, 1.5, n_obs)
    )

    return pd.DataFrame({
        'current_inflation':  current_inflation,
        'money_growth':       money_growth,
        'gdp_growth':         gdp_growth,
        'output_gap':         output_gap,
        'exchange_rate_chg':  exchange_rate_chg,
        'oil_price_chg':      oil_price_chg,
        'interest_rate':      interest_rate,
        'fiscal_deficit':     fiscal_deficit,
        'external_debt':      external_debt,
        'credit_growth':      credit_growth,
        'next_inflation':     np.clip(next_inflation, 0, 60),
    })


df = generate_macro_forecast_data()
FEATURE_COLS = [c for c in df.columns if c != 'next_inflation']
TARGET_COL   = 'next_inflation'

print(f"\nDataset : {df.shape[0]} obs  x  {df.shape[1]} variables")
print(df.describe().round(2).to_string())

## ── SECTION 3: TRAIN / TEST SPLIT

In [ ]:
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED,
)

print(f"\nTrain : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")

## ── SECTION 4: TRAIN XGBOOST

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    verbosity=0,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_pred = xgb_model.predict(X_test)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)
cv_r2  = cross_val_score(xgb_model, X, y, cv=5, scoring='r2').mean()

print(f"\n── XGBoost Performance ──────────────────────────────────────")
print(f"  RMSE (test) : {rmse:.4f} pp")
print(f"  R²   (test) : {r2:.4f}")
print(f"  R²   (5-fold CV) : {cv_r2:.4f}")

# Actual vs predicted
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=30, label='Predictions')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', linewidth=1.2, label='Perfect forecast')
ax.set_title(f'XGBoost — Actual vs Predicted Inflation  |  R² = {r2:.3f}', fontweight='bold')
ax.set_xlabel('Actual Inflation (%)')
ax.set_ylabel('Predicted Inflation (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lecture21_demo_xgb_scatter.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_xgb_scatter.png")

## ── SECTION 5: SHAP — GLOBAL FEATURE IMPORTANCE

In [ ]:
print("\nComputing SHAP values ...")

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(pd.DataFrame(X_test, columns=FEATURE_COLS))

# SHAP bar plot — mean absolute SHAP values
fig, ax = plt.subplots(figsize=(8, 5))
shap.plots.bar(shap_values, max_display=10, ax=ax, show=False)
ax.set_title('Global Feature Importance — Mean |SHAP| Value',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('lecture21_demo_shap_bar.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_shap_bar.png")

# SHAP beeswarm (summary) plot
fig, ax = plt.subplots(figsize=(9, 6))
shap.plots.beeswarm(shap_values, max_display=10, ax=ax, show=False)
ax.set_title('SHAP Summary (Beeswarm) — Feature Impact on Inflation Forecast',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('lecture21_demo_shap_beeswarm.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_shap_beeswarm.png")

## ── SECTION 6: SHAP — LOCAL EXPLANATION (WATERFALL PLOT)

Select a high-inflation and a low-inflation observation for contrast

In [ ]:
high_idx = int(np.argmax(y_test))
low_idx  = int(np.argmin(y_test))

for idx, label in [(high_idx, 'High-Inflation'), (low_idx, 'Low-Inflation')]:
    fig, ax = plt.subplots(figsize=(9, 5))
    shap.plots.waterfall(shap_values[idx], max_display=10, ax=ax, show=False)
    ax.set_title(
        f'SHAP Waterfall — {label} Observation  '
        f'(Actual = {y_test[idx]:.2f} %, Predicted = {y_pred[idx]:.2f} %)',
        fontsize=10, fontweight='bold',
    )
    plt.tight_layout()
    fname = f'lecture21_demo_shap_waterfall_{label.lower().replace("-", "_")}.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f"Saved : {fname}")

## ── SECTION 7: SHAP DEPENDENCE PLOTS

Show how SHAP value for 'current_inflation' varies with its magnitude

In [ ]:
top2_features = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=FEATURE_COLS,
).nlargest(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('SHAP Dependence Plots — Top 2 Features', fontsize=12, fontweight='bold')

for ax, feat in zip(axes, top2_features):
    feat_idx = FEATURE_COLS.index(feat)
    ax.scatter(
        X_test[:, feat_idx],
        shap_values.values[:, feat_idx],
        alpha=0.5, color='steelblue', s=25,
    )
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel(f'SHAP Value for {feat.replace("_", " ").title()}')
    ax.set_title(f'Dependence: {feat.replace("_", " ").title()}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lecture21_demo_shap_dependence.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_shap_dependence.png")

## ── SECTION 8: BOOTSTRAP CONFIDENCE INTERVALS

In [ ]:
print("\nBootstrapping prediction intervals (500 iterations) ...")

N_BOOT    = 500
boot_preds = np.zeros((N_BOOT, X_test.shape[0]))

for b in range(N_BOOT):
    idx_boot = np.random.choice(len(X_train), size=len(X_train), replace=True)
    m_boot   = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.80, colsample_bytree=0.80,
        random_state=b, verbosity=0,
    )
    m_boot.fit(X_train[idx_boot], y_train[idx_boot], verbose=False)
    boot_preds[b] = m_boot.predict(X_test)

ci_lower = np.percentile(boot_preds, 5,  axis=0)
ci_upper = np.percentile(boot_preds, 95, axis=0)
coverage = np.mean((y_test >= ci_lower) & (y_test <= ci_upper)) * 100

print(f"90 % interval coverage : {coverage:.1f} %  (target: 90 %)")

# Visualise for first 40 test observations
n_show  = 40
idx_sort = np.argsort(y_test[:n_show])

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(n_show),
                ci_lower[:n_show][idx_sort],
                ci_upper[:n_show][idx_sort],
                alpha=0.25, color='tomato', label='90 % Bootstrap CI')
ax.plot(range(n_show), y_pred[:n_show][idx_sort],
        color='tomato', linewidth=1.5, linestyle='--', label='Point Forecast')
ax.scatter(range(n_show), y_test[:n_show][idx_sort],
           color='steelblue', s=30, zorder=3, label='Actual')
ax.set_title(f'XGBoost Forecast with 90 % Bootstrap Confidence Intervals  '
             f'(coverage = {coverage:.1f} %)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Observation (sorted by actual inflation)')
ax.set_ylabel('Next-Year Inflation (%)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('lecture21_demo_bootstrap_ci.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_bootstrap_ci.png")

## ── SECTION 9: POLICY COMMUNICATION CHART

Create a clear, non-technical chart suitable for a policy brief

In [ ]:
mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=FEATURE_COLS,
).sort_values(ascending=True)

direction = []
for feat in mean_shap.index:
    fi = FEATURE_COLS.index(feat)
    corr = np.corrcoef(X_test[:, fi], shap_values.values[:, fi])[0, 1]
    direction.append('Upward pressure' if corr > 0 else 'Downward pressure')

bar_colors = ['#F44336' if d == 'Upward pressure' else '#4CAF50' for d in direction]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(mean_shap.index, mean_shap.values, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_title('What Drives Inflation Forecasts?\nAverage Impact of Each Factor',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Average Influence on Forecast (SHAP Value, percentage points)')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F44336', label='Typically raises inflation'),
    Patch(facecolor='#4CAF50', label='Typically lowers inflation'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

for bar in bars:
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('lecture21_demo_policy_chart.png', bbox_inches='tight')
plt.show()
print("Saved : lecture21_demo_policy_chart.png")

print("""
── Policy Narrative (example for a non-technical briefing) ─────────────────
 "Our model analysed 500 country-year observations to forecast next-year
  inflation. The single most important driver is current inflation — economies
  with high inflation today are very likely to have high inflation next year.
  Money supply growth is the second strongest driver: a 10 percentage-point
  increase in money supply growth adds approximately 3 pp to the inflation
  forecast. By contrast, higher GDP growth and interest rates tend to reduce
  inflationary pressure. These findings are consistent with standard
  macroeconomic theory and provide a transparent, auditable basis for
  central bank inflation projections."
────────────────────────────────────────────────────────────────────────────
""")

## ── SECTION 10: SUMMARY

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  LECTURE 21 DEMO COMPLETE                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║  Concepts demonstrated:                                           ║
║   1. XGBoost regression for inflation forecasting                 ║
║   2. SHAP TreeExplainer — global & local interpretability         ║
║   3. SHAP bar & beeswarm plots (global importance)                ║
║   4. SHAP waterfall plots (local / single-observation)            ║
║   5. SHAP dependence plots (non-linear feature effects)           ║
║   6. Bootstrap confidence intervals (90 % coverage)               ║
║   7. Policy communication chart for non-technical audiences       ║
╠═══════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                   ║
║   lecture21_demo_xgb_scatter.png                                  ║
║   lecture21_demo_shap_bar.png                                     ║
║   lecture21_demo_shap_beeswarm.png                                ║
║   lecture21_demo_shap_waterfall_high_inflation.png                ║
║   lecture21_demo_shap_waterfall_low_inflation.png                 ║
║   lecture21_demo_shap_dependence.png                              ║
║   lecture21_demo_bootstrap_ci.png                                 ║
║   lecture21_demo_policy_chart.png                                 ║
╚═══════════════════════════════════════════════════════════════════╝
""")